In [ ]:
# Imports
from datetime import datetime, timezone
from pathlib import Path
import json
import pandas as pd

from src.parte_01_dados.quality import build_run_manifest, quality_status
from src.parte_01_dados.silver import EXPECTED_COLUMNS, build_processing_metrics, clean_calls, validate_columns

BRONZE_ROOT = Path("/Volumes/workspace/conectatel/raw_files/bronze")
SILVER_ROOT = Path("/Volumes/workspace/conectatel/raw_files/silver")
SILVER_ROOT.mkdir(parents=True, exist_ok=True)
SNAPSHOT_PATH = BRONZE_ROOT / "bronze_calls_snapshot.csv"
CLEANED_PATH = SILVER_ROOT / "silver_calls_cleaned.csv"
QUALITY_PATH = SILVER_ROOT / "silver_quality_report.json"
SCHEMA_PATH = SILVER_ROOT / "silver_schema.json"
METRICS_PATH = SILVER_ROOT / "silver_processing_metrics.json"


In [ ]:
# Leitura
if not SNAPSHOT_PATH.exists():
    raise FileNotFoundError(f"Snapshot não encontrado: {SNAPSHOT_PATH}")
started_at = datetime.now(timezone.utc)
calls = pd.read_csv(SNAPSHOT_PATH, dtype="string", keep_default_na=True)
validate_columns(calls)
input_rows = len(calls)
print(f"Linhas brutas: {input_rows}")
print(f"Colunas: {len(calls.columns)}")


In [ ]:
# Limpeza
cleaned = clean_calls(calls)
output_rows = len(cleaned)
removed_rows = input_rows - output_rows
print(f"Linhas depois: {output_rows}")
print(f"Duplicatas removidas: {removed_rows}")

raw_dates = pd.to_datetime(calls["data_abertura"], errors="coerce", format="mixed")
raw_durations = pd.to_numeric(calls["duracao_minutos"], errors="coerce")
raw_satisfaction = pd.to_numeric(calls["satisfacao_1_a_5"], errors="coerce")
quality = {
    "input_rows": input_rows,
    "output_rows": output_rows,
    "exact_duplicates_removed": removed_rows,
    "invalid_dates": int((calls["data_abertura"].notna() & raw_dates.isna()).sum()),
    "invalid_durations": int((calls["duracao_minutos"].notna() & raw_durations.isna()).sum()),
    "invalid_satisfaction": int((calls["satisfacao_1_a_5"].notna() & raw_satisfaction.isna()).sum()),
}


In [ ]:
# Persistência
cleaned.to_csv(CLEANED_PATH, index=False)
finished_at = datetime.now(timezone.utc)
metrics = build_processing_metrics(input_rows=input_rows, output_rows=output_rows, started_at=started_at, finished_at=finished_at)
run_status = quality_status(output_rows=output_rows, invalid_values=sum(quality[key] for key in ["invalid_dates", "invalid_durations", "invalid_satisfaction"]), duplicate_rows=removed_rows)
manifest = build_run_manifest(layer="silver", source_path=SNAPSHOT_PATH, output_paths=[CLEANED_PATH, QUALITY_PATH, SCHEMA_PATH, METRICS_PATH], input_rows=input_rows, output_rows=output_rows, status=run_status, started_at=started_at, finished_at=finished_at)
quality.update({"status": run_status, "run_manifest": manifest})
QUALITY_PATH.write_text(json.dumps(quality, ensure_ascii=False, indent=2), encoding="utf-8")
SCHEMA_PATH.write_text(json.dumps({"columns": EXPECTED_COLUMNS, "dtypes": {col: str(cleaned[col].dtype) for col in cleaned.columns}}, indent=2), encoding="utf-8")
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Status: {run_status}")


In [ ]:
# Metadados
metadata_source = BRONZE_ROOT / "bronze_corpus_metadata.json"
metadata_target = SILVER_ROOT / "silver_corpus_metadata.json"
if metadata_source.exists():
    metadata_target.write_text(metadata_source.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Metadados copiados: {metadata_target}")
else:
    print("Metadados da Bronze não encontrados")
